# phonon-omega-max — single-fold walkthrough

This notebook runs one outer CV fold end-to-end:

1. Load Matbench `phonons` (cached after first call).
2. Featurize: Magpie composition + CGCNN graphs.
3. Train GBDT and one CGCNN seed on the fold's training split.
4. Evaluate on the fold's test split.
5. Show parity + residual plot for the single fold.

The full 5-fold × 5-seed result is produced by `make all` at the
command line; this notebook is for getting your hands on the data and
the models in one shot.


In [ ]:
from pathlib import Path
from phonon_omegamax.data.load import load_phonons
from phonon_omegamax.data.split import kfold_indices, inner_train_val_split
from phonon_omegamax.features.magpie import featurize_samples
from phonon_omegamax.models.dataset import StructureDataset
from phonon_omegamax.models.train import train_one_seed
from phonon_omegamax.models.gbdt import GBDTRegressor
import numpy as np

ROOT = Path.cwd().parent
samples = load_phonons(cache_path=ROOT / 'data' / 'cache' / 'phonons.parquet')
print(f'{len(samples)} samples loaded')


In [ ]:
folds = list(kfold_indices(len(samples), seed=42))
train_idx, test_idx = folds[0]
inner_train, inner_val = inner_train_val_split(train_idx, val_frac=0.2, seed=0)
print(f'fold 0: train={len(inner_train)} val={len(inner_val)} test={len(test_idx)}')

X = featurize_samples(samples, cache_path=ROOT / 'data' / 'cache' / 'magpie.npy')
y = np.array([s.target for s in samples])


In [ ]:
gbdt = GBDTRegressor(random_state=42).fit(X[train_idx], y[train_idx])
gbdt_pred = gbdt.predict(X[test_idx])
print(f'GBDT fold-0 MAE = {np.abs(gbdt_pred - y[test_idx]).mean():.1f} cm-1')


In [ ]:
train_ds = StructureDataset(
    [samples[i] for i in inner_train],
    cache_dir=ROOT / 'data' / 'cache' / 'graphs',
)
val_ds = StructureDataset(
    [samples[i] for i in inner_val],
    cache_dir=ROOT / 'data' / 'cache' / 'graphs',
)
test_ds = StructureDataset(
    [samples[i] for i in test_idx],
    cache_dir=ROOT / 'data' / 'cache' / 'graphs',
)

result = train_one_seed(
    train_ds, val_ds, epochs=50, batch_size=32, seed=0,
    ckpt_path=ROOT / 'checkpoints' / 'walkthrough_fold0_seed0.pt',
)
print(f'best val MAE = {result.best_val_mae:.1f} after {result.epochs_run} epochs')


In [ ]:
import torch
from torch_geometric.loader import DataLoader
from phonon_omegamax.models.cgcnn import CGCNN

ckpt = torch.load(
    ROOT / 'checkpoints' / 'walkthrough_fold0_seed0.pt',
    map_location='cpu', weights_only=False,
)
model = CGCNN()
model.load_state_dict(ckpt['state_dict'])
model.eval()

loader = DataLoader(test_ds, batch_size=32, shuffle=False)
cgcnn_pred = []
with torch.no_grad():
    for batch in loader:
        cgcnn_pred.append(model(batch).cpu().numpy())
cgcnn_pred = np.concatenate(cgcnn_pred)
print(f'CGCNN fold-0 MAE = {np.abs(cgcnn_pred - y[test_idx]).mean():.1f} cm-1')


In [ ]:
from phonon_omegamax.eval.parity import headline_figure
headline_figure(
    y_true=y[test_idx], gbdt_pred=gbdt_pred, cgcnn_pred=cgcnn_pred,
)


## Next steps

Run `make all` at the command line to reproduce the full 5-fold ×
5-seed result and the headline figure that backs the leaderboard
table.
